# Train LookupClassifier with Hard Labels

Fit a `LookupClassifier` in `label_mode="hard"` using the `label` column (40 classes, including background) and save the fitted model to disk.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import sys
import os

sys.path.append(str(Path(os.getcwd())))
sys.path.append(str(Path(os.getcwd()).parent))
from methyldl.modelling.classifiers.lookup import LookupClassifier, LabelConfig

%load_ext autoreload
%autoreload 2

['/apps/leuven/rocky8/skylake/2025a/software/Python/3.13.1-GCCcore-14.2.0/lib/python313.zip', '/apps/leuven/rocky8/skylake/2025a/software/Python/3.13.1-GCCcore-14.2.0/lib/python3.13', '/apps/leuven/rocky8/skylake/2025a/software/Python/3.13.1-GCCcore-14.2.0/lib/python3.13/lib-dynload', '', '/data/leuven/389/vsc38912/Projects/methyldl/.venv-P100/lib/python3.13/site-packages', '/vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl/EDA', '/vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl/EDA', '/vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl']


## 1. Load training data

In [ ]:
DATA_DIR = Path(
    "../Data/training_data/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041"
)

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
print(f"Train: {len(train_df):,} reads, {train_df['name'].nunique()} regions")
print(f"Label range: {train_df['label'].min()} - {train_df['label'].max()}")

## 2. Configure and fit

In [ ]:
config = LabelConfig(
    num_classes=40,
    label_col="label",
    label_mode="hard",
)
print(config)

In [ ]:
clf = LookupClassifier(config)
clf.fit(train_df)

## 3. Save fitted classifier

In [ ]:
OUTPUT_DIR = Path("../output/lookup_hard_labels")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_path = OUTPUT_DIR / "lookup_hard_fitted.pkl"

In [ ]:
clf.save(save_path)
print(f"Saved to {save_path} ({save_path.stat().st_size / 1e6:.1f} MB)")

## 4. Quick verification: reload and spot-check

In [ ]:
clf2 = LookupClassifier.load(save_path)
print(f"Reloaded config: {clf2.config}")
print(f"Lookup keys: {len(clf2._lookup):,}")

# Spot-check a single entry
sample_key = next(iter(clf2._lookup))
entry = clf2._lookup[sample_key]
print(f"\nSample key: {sample_key[0]}")
print(f"Hard label (argmax): {entry['hard_label'].index(max(entry['hard_label']))}")
print(f"Hard label vector sum: {sum(entry['hard_label']):.4f}")

## 5. Compute MSE on train set

Predict on the training data and compute the MSE between predictions and one-hot encoded targets.

In [ ]:
# Predict on train set
train_pred = clf.predict(train_df)

# Build one-hot targets from label column
num_classes = config.num_classes
targets = np.zeros((len(train_pred), num_classes))
targets[np.arange(len(train_pred)), train_pred[config.label_col].values] = 1.0

# Extract prediction matrix
pred_cols = [f"prediction_{j}" for j in range(num_classes)]
preds = train_pred[pred_cols].values

# Compute MSE
mse = np.mean((preds - targets) ** 2)
print(f"Train MSE: {mse:.6f}")

# Also per-sample MSE stats
per_sample_mse = np.mean((preds - targets) ** 2, axis=1)
print(
    f"Per-sample MSE — mean: {per_sample_mse.mean():.6f}, median: {np.median(per_sample_mse):.6f}, max: {per_sample_mse.max():.6f}"
)

# Breakdown by prediction source
for source in ["exact", "1nn"]:
    mask = train_pred["prediction_source"] == source
    if mask.any():
        source_mse = np.mean((preds[mask] - targets[mask]) ** 2)
        print(f"  {source}: MSE={source_mse:.6f} ({mask.sum():,} reads)")

## 6. Compute MSE on val set

In [ ]:
val_df = pd.read_parquet(DATA_DIR / "valid.parquet")
print(f"Validation: {len(val_df):,} reads, {val_df['name'].nunique()} regions")
print(f"Label range: {val_df['label'].min()} - {val_df['label'].max()}")

In [ ]:
# Predict on train set
val_pred = clf2.predict(val_df)

In [ ]:
# Build one-hot targets from label column
num_classes = config.num_classes
targets = np.zeros((len(val_pred), num_classes))
targets[np.arange(len(val_pred)), val_pred[config.label_col].values] = 1.0

# Extract prediction matrix
pred_cols = [f"prediction_{j}" for j in range(num_classes)]
preds = val_pred[pred_cols].values

# Compute MSE
mse = np.mean((preds - targets) ** 2)
print(f"Validation MSE: {mse:.6f}")

# Also per-sample MSE stats
per_sample_mse = np.mean((preds - targets) ** 2, axis=1)
print(
    f"Per-sample MSE — mean: {per_sample_mse.mean():.6f}, median: {np.median(per_sample_mse):.6f}, max: {per_sample_mse.max():.6f}"
)

# Breakdown by prediction source
for source in ["exact", "1nn"]:
    mask = val_pred["prediction_source"] == source
    if mask.any():
        source_mse = np.mean((preds[mask] - targets[mask]) ** 2)
        print(f"  {source}: MSE={source_mse:.6f} ({mask.sum():,} reads)")

## 7. Train on cfsort setting

In [2]:
train_cfsort_df = pd.read_parquet(
    "/staging/leuven/stg_00118/methylDL/data/HG19_ForRRBSsplits_LOOKUPONLY_205files/train.parquet"
)

In [3]:
print(
    f"Train: {len(train_cfsort_df):,} reads, {train_cfsort_df['name'].nunique()} regions"
)
print(
    f"Label range: {train_cfsort_df['label'].min()} - {train_cfsort_df['label'].max()}"
)

Train: 2,566,332 reads, 811 regions
Label range: 0 - 39


In [4]:
config = LabelConfig(
    num_classes=40,
    label_col="label",
    label_mode="hard",
)
print(config)

LabelConfig(num_classes=40, label_col='label', label_mode='hard', min_reads=30, max_distance=0.41)


In [6]:
clf_cfsort = LookupClassifier(config)
_ = clf_cfsort.fit(train_cfsort_df)

Extracting signatures: 100%|██████████| 2566332/2566332 [00:39<00:00, 65131.36it/s]


In [7]:
clf_cfsort.save(
    "/staging/leuven/stg_00118/methylDL/experiments/cfsort_rrbs/Hg19_LookupClassifier_HardLabels/lookup_hard_gh19_cfsort_rrbs.pkl"
)